# LGD Experiments: Uniform · Bimodal · Unimodal

Loads pretrained models from HuggingFace and runs LGD optimization with **seeds 0–15**.

Each cell is self-contained — run them top to bottom. Results display live after every seed.

## 1 · Setup & Imports

In [ ]:
# Install dependencies (only needed once in a fresh Colab runtime)
!pip install -q diffusers huggingface_hub accelerate

In [ ]:
import os, math
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.distributions import Categorical, MultivariateNormal, MixtureSameFamily

from diffusers import DDPMScheduler, UNet2DModel, DDIMScheduler
from huggingface_hub import hf_hub_download
from tqdm.notebook import tqdm

device = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)
print(f'Device: {device}')

In [ ]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

## 2 · HuggingFace Login

Add your `HF_TOKEN` to **Colab Secrets** (🔑 icon in the left sidebar) then run this cell.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF')
print('HF_TOKEN loaded ✓')

## 3 · Model Definitions

In [ ]:
# ── Shared building blocks ────────────────────────────────────────────────

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=1)


class GenericNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t_emb=None):
        if t_emb is not None and t_emb.dim() == 2 and x.dim() == 2:
            x = x + t_emb
        return self.net(x)

In [ ]:
# ── Conditional model ─────────────────────────────────────────────────────

class CircularAngleConsistencyModel(nn.Module):
    """Conditional iCT model predicting rotation angle as (cos, sin)."""

    def __init__(self, nfeatures=2, img_features=784, eps=0.002,
                 nunits=128, depth=6, device=None):
        super().__init__()
        self.eps          = eps
        self.nfeatures    = nfeatures
        self.img_features = img_features
        self.nunits       = nunits
        self.device       = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.betas        = None

        self.cond_embed = nn.Sequential(
            nn.Unflatten(1, (1, 28, 28)),
            nn.Conv2d(1, 32, 3, padding=1),
            nn.GroupNorm(8, 32), nn.SiLU(), nn.Dropout2d(0.1),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.GroupNorm(8, 32), nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.GroupNorm(8, 64), nn.SiLU(), nn.Dropout2d(0.15),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.GroupNorm(8, 64), nn.SiLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64 * 7 * 7, nunits),
            nn.SiLU(),
        )
        self.input_layer = nn.Linear(nfeatures, nunits)
        self.input_norm  = nn.LayerNorm(nunits)
        self.output_norm = nn.LayerNorm(nunits)
        self.time_embed  = TimeEmbedding(nunits)
        self.net         = GenericNN(nunits, [nunits] * depth, nunits)
        self.out         = nn.Linear(nunits, nfeatures)
        self.c_huber     = 0.00054 * math.sqrt(nfeatures)
        self.to(self.device)

    def forward(self, x, t, cond=None):
        x_ori = x
        x = self.input_norm(self.input_layer(x))
        if cond is not None:
            if cond.dim() == 4:   cond = cond.view(cond.size(0), -1)
            elif cond.dim() == 3: cond = cond.view(cond.size(0), -1)
            x = x + self.cond_embed(cond)
        if isinstance(t, (float, int)):
            t = torch.tensor([t] * x.shape[0], dtype=torch.float32, device=x.device).unsqueeze(1)
        elif t.dim() == 1: t = t.unsqueeze(1)
        elif t.dim() == 3: t = t.squeeze(-1)
        x = x + self.time_embed(t.squeeze(-1))
        x = self.output_norm(self.net(x))
        x = self.out(x)
        t_w    = t - self.eps
        c_skip = 0.25 / (t_w.pow(2) + 0.25)
        c_out  = 0.25 * t_w / ((t_w + self.eps).pow(2) + 0.25).sqrt()
        result = c_skip * x_ori + c_out * x
        return result / (torch.norm(result, dim=1, keepdim=True) + 1e-8)

    def sample(self, nsamples=250, condition_x=None,
               ts: List[float] = [150.0, 50.0, 20.0, 10.0, 5.0, 1.], device=None):
        device = self.device if device is None else device
        if condition_x is not None:
            condition_x = condition_x.to(device)
            if condition_x.dim() > 2:
                condition_x = condition_x.view(condition_x.size(0), -1)
        x = torch.randn(nsamples, self.nfeatures, device=device) * ts[0]
        x = x / (torch.norm(x, dim=1, keepdim=True) + 1e-8)
        for t in ts[1:]:
            z = torch.randn_like(x)
            x = x + math.sqrt(t ** 2 - self.eps ** 2) * z
            x = self(x, t, cond=condition_x)
        return x, None, None

In [ ]:
# ── Unconditional model ───────────────────────────────────────────────────

class UnconditionalUnet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = UNet2DModel(
            sample_size=28,
            in_channels=1, out_channels=1,
            layers_per_block=2,
            block_out_channels=(32, 64, 128),
            down_block_types=('DownBlock2D', 'AttnDownBlock2D', 'AttnDownBlock2D'),
            up_block_types=('AttnUpBlock2D', 'AttnUpBlock2D', 'UpBlock2D'),
            dropout=0.1,
        )

    def forward(self, x, t):
        return self.model(x, t).sample

## 4 · Utilities

In [ ]:
# ── Angle conversion ──────────────────────────────────────────────────────

def angles_to_circular(angles_deg):
    rad = torch.deg2rad(angles_deg.float())
    return torch.stack([torch.cos(rad), torch.sin(rad)], dim=-1)

def circular_to_angles(circular):
    return torch.rad2deg(torch.atan2(circular[..., 1], circular[..., 0])) % 360


# ── MoG helpers ───────────────────────────────────────────────────────────

def mog_pdf(x, means, variances, weights=None):
    components = len(means)
    if weights is None:
        weights = torch.ones(components) / components
    pdf = torch.zeros_like(x)
    for mean, var, weight in zip(means, variances, weights):
        var_t  = torch.tensor(var,  dtype=torch.float32)
        mean_t = torch.tensor(mean, dtype=torch.float32)
        diff   = (x - mean_t + 180) % 360 - 180
        pdf   += weight * torch.exp(-0.5 * diff**2 / var_t) / (
            torch.sqrt(2 * torch.pi * var_t)
        )
    return pdf

def create_mog_pdf_evaluator(mog_means, mog_variances, weights):
    def evaluate_pdf(angles):
        return mog_pdf(
            angles,
            [m.item() for m in mog_means],
            [v.squeeze().item() for v in mog_variances],
            weights,
        )
    return evaluate_pdf

def generate_mog_samples(num_samples, means, variances, weights=None, device='cpu'):
    components = len(means)
    if weights is None:
        weights = torch.ones(components, device=device) / components
    else:
        weights = weights.to(device)
    weights = weights / weights.sum()
    flattened  = [m.flatten() for m in means]
    dim        = flattened[0].shape[0]
    means_t    = torch.stack(flattened).to(device)
    if variances[0].numel() == dim * dim:
        covs_t = torch.stack([v.reshape(dim, dim) for v in variances]).to(device)
    else:
        covs_t = torch.stack([torch.diag(v.flatten()) for v in variances]).to(device)
    mix     = Categorical(weights)
    comp    = MultivariateNormal(means_t, covs_t)
    mixture = MixtureSameFamily(mix, comp)
    return mixture.sample((num_samples,))


# ── Loss (Sliced Wasserstein) ─────────────────────────────────────────────

def sliced_wasserstein_distance(X, Y, n_projections=50, device='cpu'):
    X = X.to(device).float()
    Y = Y.to(device).float()
    dim  = X.shape[1]
    proj = torch.randn(n_projections, dim, device=device)
    proj = proj / torch.norm(proj, dim=1, keepdim=True)
    X_s  = torch.sort(X @ proj.T, dim=0)[0]
    Y_s  = torch.sort(Y @ proj.T, dim=0)[0]
    return torch.mean(torch.abs(X_s - Y_s))


# ── Seed ──────────────────────────────────────────────────────────────────

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

## 5 · Load Pretrained Models

In [ ]:
hf_token = os.environ.get('HF_TOKEN')

print('Downloading conditional model...')
cond_path = hf_hub_download(
    repo_id='Orineo/conditional-matching-paper',
    filename='MNIST/MnistConditional500Epoch.pt',
    token=hf_token,
)

print('Downloading unconditional model...')
uncond_path = hf_hub_download(
    repo_id='Orineo/conditional-matching-paper',
    filename='MNIST/MnistUncond100Epoch.pth',
    token=hf_token,
)

# Conditional
cond_model = CircularAngleConsistencyModel(
    nfeatures=2, img_features=784, eps=0.002, nunits=128, depth=5, device=device,
)
ckpt = torch.load(cond_path, map_location=device)
cond_model.load_state_dict(ckpt['model_state_dict'])
cond_model.eval()
print(f'Conditional model loaded  (epoch {ckpt["epoch"]})')

# Unconditional
uncond_model = UnconditionalUnet().to(device)
ckpt_u = torch.load(uncond_path, map_location=device)
uncond_model.load_state_dict(ckpt_u['model_state_dict'])
uncond_model.eval()
print(f'Unconditional model loaded (epoch {ckpt_u["epoch"]})')

# Noise scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule='squaredcos_cap_v2')
print('Ready ✓')

## 6 · LGD Core Function

In [ ]:
def optimize_LGD(model_uncond, model_cond_cm, noise_scheduler,
                 mog_means, mog_variances, weights,
                 nsamples=500, num_x_t=10, device='cuda',
                 lr=0.01, use_uniform=False, verbose=True, seed=None):  # ← add seed

    # Re-set seed here, not outside — guarantees identical RNG state
    # regardless of what ran before this call
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

    ddim = DDIMScheduler.from_config(noise_scheduler.config)
    ddim.set_timesteps(num_inference_steps=100)
    timesteps = ddim.timesteps

    pdf_eval = (
        (lambda x: torch.ones_like(x) / 360.0)
        if use_uniform
        else create_mog_pdf_evaluator(mog_means, mog_variances, weights)
    )

    x_t = torch.randn(1, 1, 28, 28, device=device, requires_grad=True)

    iterator = tqdm(enumerate(timesteps[:-1]), total=len(timesteps)-1, leave=False) \
               if verbose else enumerate(timesteps[:-1])

    for i, t in iterator:
        x_t       = x_t.detach().clone().requires_grad_(True)
        optimizer = optim.Adam([x_t], lr=lr)

        residual       = model_uncond(x_t, torch.tensor([t], device=device))
        alpha_t        = ddim.alphas_cumprod[t]
        alpha_t_prev   = (ddim.alphas_cumprod[timesteps[i+1]]
                          if i < len(timesteps)-2 else torch.tensor(1.0))
        beta_t         = 1 - alpha_t
        pred_x0        = (x_t - beta_t**0.5 * residual) / alpha_t**0.5
        x_t_minus_1    = alpha_t_prev**0.5 * pred_x0 + (1 - alpha_t_prev)**0.5 * residual

        r_t       = torch.sqrt(beta_t)
        step_size = r_t / (1 + r_t**2) + 5 * t / 1000

        losses = []
        for _ in range(num_x_t):
            x0_sample     = pred_x0 + r_t**2 * torch.randn_like(pred_x0)
            target_angles = circular_to_angles(
                model_cond_cm.sample(nsamples=nsamples, condition_x=x0_sample,
                                     ts=[150., 50., 20., 10., 5., 1.])[0]
            )
            target_circ = angles_to_circular(target_angles)

            if use_uniform:
                mog_circ = angles_to_circular(torch.rand(nsamples, device=device) * 360)
            else:
                mog_ang  = generate_mog_samples(nsamples, mog_means, mog_variances,
                                                weights).squeeze()
                mog_circ = angles_to_circular(mog_ang)

            loss_val = sliced_wasserstein_distance(target_circ, mog_circ,
                                                   n_projections=50, device=device)
            losses.append(-loss_val)

        log_me = -torch.logsumexp(torch.stack(losses), dim=0) + math.log(num_x_t)

        if verbose:
            iterator.set_postfix(t=t.item(), loss=f'{log_me.item():.4f}')

        grad = torch.autograd.grad(log_me, x_t, retain_graph=True)[0]
        with torch.no_grad():
            if t < 200:
                step_size = 0
            x_t = x_t_minus_1.detach().clone() - step_size * grad

    # Final DDIM step
    with torch.no_grad():
        last_t  = timesteps[-1]
        res     = model_uncond(x_t, torch.tensor([last_t], device=device))
        a       = ddim.alphas_cumprod[last_t]
        x_t     = (x_t - (1 - a)**0.5 * res) / a**0.5

    x_final = x_t.detach().clone()

    # Final evaluation loss (4× samples)
    final_n     = nsamples * 4
    final_ang   = circular_to_angles(
        model_cond_cm.sample(nsamples=final_n, condition_x=x_final.view(1, 28, 28),
                             ts=[150., 50., 20., 10., 5., 1.])[0]
    )
    final_circ  = angles_to_circular(final_ang)
    if use_uniform:
        ref_circ = angles_to_circular(torch.rand(final_n, device=device) * 360)
    else:
        ref_ang  = generate_mog_samples(final_n, mog_means, mog_variances, weights).squeeze()
        ref_circ = angles_to_circular(ref_ang)

    final_loss = sliced_wasserstein_distance(final_circ, ref_circ, n_projections=50, device=device)

    if device == 'cuda':
        torch.cuda.empty_cache()

    return x_final, final_loss

## 7 · Experiment Runner

Displays each generated image immediately, then shows a ranked summary grid after all seeds.

In [ ]:
def plot_top_k_images(results, loss_log, seed_log, experiment_name, top_k=10):
    """Plot ranked grid of top-k generated images."""
    k      = min(top_k, len(results))
    top_ix = np.argsort(loss_log)[:k]
    ncols  = min(5, k)
    nrows  = math.ceil(k / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
    axes = np.array(axes).reshape(nrows, ncols)
    for rank, idx in enumerate(top_ix):
        r, c = divmod(rank, ncols)
        axes[r, c].imshow(results[idx], cmap='gray')
        axes[r, c].set_title(f'Rank {rank+1} | Loss {loss_log[idx]:.4f}\nSeed {seed_log[idx]}', fontsize=9)
        axes[r, c].axis('off')
    for rank in range(k, nrows * ncols):
        r, c = divmod(rank, ncols)
        axes[r, c].axis('off')
    plt.suptitle(f'[{experiment_name}] Top {k} Generated Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return top_ix  # return so dist plot can reuse the ranking


def plot_top_k_distributions(results, loss_log, seed_log, model_cond,
                              target_pdf, x_range, experiment_name,
                              top_ix, top_k_dist=5):
    """Plot angle distributions for the top-k_dist results (shared y-axis)."""
    dist_k  = min(top_k_dist, len(top_ix))
    dist_ix = top_ix[:dist_k]

    all_max_y = target_pdf.max().item()
    temp_angs = []
    for idx in dist_ix:
        cond_t = torch.tensor(results[idx], dtype=torch.float32).flatten().unsqueeze(0)
        ang    = circular_to_angles(
            model_cond.sample(nsamples=500, condition_x=cond_t,
                              ts=[150., 50., 20., 10., 5., 1.])[0]
        )
        temp_angs.append(ang)
        h, _ = np.histogram(ang.detach().cpu().numpy(), bins=30, range=(0, 360), density=True)
        all_max_y = max(all_max_y, h.max() if len(h) else 0)
    y_lim = all_max_y * 1.1

    ncols_d = min(5, dist_k)
    nrows_d = math.ceil(dist_k / ncols_d)

    fig, axes = plt.subplots(nrows_d, ncols_d, figsize=(ncols_d * 5, nrows_d * 4))
    axes = np.array(axes).reshape(nrows_d, ncols_d)
    for rank, (idx, ang) in enumerate(zip(dist_ix, temp_angs)):
        r, c = divmod(rank, ncols_d)
        ax   = axes[r, c]
        ax.hist(ang.detach().cpu().numpy(), bins=30, alpha=0.6,
                color='skyblue', edgecolor='black', range=(0, 360),
                density=True, label='Sampled')
        ax.plot(x_range.numpy(), target_pdf.numpy(),
                color='orange', linewidth=2, label='Target')
        ax.set_title(f'Rank {rank+1} | Loss {loss_log[idx]:.4f}\nSeed {seed_log[idx]}', fontsize=9)
        ax.set_xlim(0, 360)
        ax.set_ylim(0, y_lim)
        ax.grid(True, alpha=0.3)
        if rank == 0:
            ax.legend(fontsize=8)
    for rank in range(dist_k, nrows_d * ncols_d):
        r, c = divmod(rank, ncols_d)
        axes[r, c].axis('off')
    plt.suptitle(f'[{experiment_name}] Top {dist_k} Angle Distributions', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def run_experiment(model_uncond, model_cond, noise_scheduler,
                   mog_means, mog_variances, weights,
                   experiment_name, seeds=range(16),
                   nsamples=500, num_x_t=10, lr=0.01,
                   use_uniform=False, top_k=5, top_k_dist=5, device='cuda'):
    """
    Run LGD for each seed, display results live, then show top-k summary.
    Image grid shows top_k results, distribution grid shows top_k_dist results.
    """
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {experiment_name}')
    print(f'{"="*60}')

    x_range = torch.linspace(0, 360, 200)
    if use_uniform:
        target_pdf = torch.ones_like(x_range) / 360.0
    else:
        target_pdf = mog_pdf(
            x_range,
            [m.item() for m in mog_means],
            [v.squeeze().item() for v in mog_variances],
            weights,
        )

    results, loss_log, seed_log = [], [], []

    for seed in seeds:
        set_seed(seed)
        print(f'\n[Seed {seed}] Running optimization...')

        x_final, loss = optimize_LGD(
            model_uncond=model_uncond,
            model_cond_cm=model_cond,
            noise_scheduler=noise_scheduler,
            mog_means=mog_means,
            mog_variances=mog_variances,
            weights=weights,
            nsamples=nsamples,
            num_x_t=num_x_t,
            device=device,
            lr=lr,
            use_uniform=use_uniform,
            verbose=True,
            seed=seed,
        )

        img      = x_final.squeeze().cpu().numpy()
        loss_val = loss.item()
        results.append(img)
        loss_log.append(loss_val)
        seed_log.append(seed)

        # ── Live result: image + angle distribution side by side ──
        cond_t  = torch.tensor(img, dtype=torch.float32).flatten().unsqueeze(0)
        ang_out = circular_to_angles(
            model_cond.sample(nsamples=500, condition_x=cond_t,
                              ts=[150., 50., 20., 10., 5., 1.])[0]
        )
        ang_np = ang_out.detach().cpu().numpy()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.imshow(img, cmap='gray')
        ax1.set_title(f'[{experiment_name}] Seed {seed}\nLoss: {loss_val:.4f}', fontsize=11)
        ax1.axis('off')

        ax2.hist(ang_np, bins=30, alpha=0.6, color='skyblue',
                 edgecolor='black', range=(0, 360), density=True, label='Sampled')
        ax2.plot(x_range.numpy(), target_pdf.numpy(),
                 color='orange', linewidth=2, label='Target')
        ax2.set_xlim(0, 360)
        ax2.set_xlabel('Angle (°)')
        ax2.set_ylabel('Density')
        ax2.set_title('Angle Distribution')
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        print(f'  → Loss: {loss_val:.4f}')

    # ── Final summary ─────────────────────────────────────────────────────
    top_ix = plot_top_k_images(results, loss_log, seed_log, experiment_name, top_k)
    plot_top_k_distributions(results, loss_log, seed_log, model_cond,
                             target_pdf, x_range, experiment_name,
                             top_ix, top_k_dist)

    return {'images': results, 'losses': loss_log, 'seeds': seed_log}

## 8 · Experiment 1 — Uniform Distribution

Target: angles uniformly distributed over [0°, 360°).

In [ ]:
results_uniform = run_experiment(
    model_uncond    = uncond_model,
    model_cond      = cond_model,
    noise_scheduler = noise_scheduler,
    mog_means       = [torch.tensor([180], dtype=torch.float64)],   # unused
    mog_variances   = [torch.tensor([[60]], dtype=torch.float64)],
    weights         = torch.tensor([1.0], dtype=torch.float64),
    experiment_name = 'Uniform',
    seeds           = range(16),          # seeds 0–15
    nsamples        = 600,
    num_x_t         = 10,
    use_uniform     = True,
    top_k           = 5,
    device          = device,
)

## 9 · Experiment 2 — Bimodal Distribution

Target: mixture of Gaussians centred at **180°** and **360°**.

In [ ]:
bimodal_means     = [torch.tensor([180], dtype=torch.float64),
                     torch.tensor([360], dtype=torch.float64)]
bimodal_variances = [torch.tensor([[260]], dtype=torch.float64)] * 2
bimodal_weights   = torch.tensor([0.5, 0.5], dtype=torch.float64)

results_bimodal = run_experiment(
    model_uncond    = uncond_model,
    model_cond      = cond_model,
    noise_scheduler = noise_scheduler,
    mog_means       = bimodal_means,
    mog_variances   = bimodal_variances,
    weights         = bimodal_weights,
    experiment_name = 'Bimodal (180° & 360°)',
    seeds           = range(16),
    nsamples        = 600,
    num_x_t         = 10,
    use_uniform     = False,
    top_k           = 5,
    device          = device,
)

## 10 · Experiment 3 — Unimodal Distribution

Target: single Gaussian centred at **360°** (≡ 0°).

In [ ]:
unimodal_means     = [torch.tensor([360], dtype=torch.float64)]
unimodal_variances = [torch.tensor([[620]], dtype=torch.float64)]
unimodal_weights   = torch.tensor([1.0], dtype=torch.float64)

results_unimodal = run_experiment(
    model_uncond    = uncond_model,
    model_cond      = cond_model,
    noise_scheduler = noise_scheduler,
    mog_means       = unimodal_means,
    mog_variances   = unimodal_variances,
    weights         = unimodal_weights,
    experiment_name = 'Unimodal (360°)',
    seeds           = range(16),
    nsamples        = 600,
    num_x_t         = 10,
    use_uniform     = False,
    top_k           = 5,
    device          = device,
)